# NYC MTA Turnstile Ridership Analysis

**Author:** Aidan Moran  
**Date:** May 2026  
**Tools:** Python, Pandas, Matplotlib, Seaborn, Folium

## Overview

The MTA publishes weekly turnstile audit data for every turnstile unit in the NYC subway system. Each record captures cumulative entry and exit counts at roughly 4-hour intervals, giving us granular ridership data across 470+ stations.

In this notebook, we'll:
1. **Download and clean** recent turnstile data
2. **Analyze ridership patterns** by time of day, day of week, and station
3. **Visualize post-pandemic recovery** trends
4. **Map station-level ridership** to identify the busiest corridors
5. **Surface insights** relevant to transit planning and operations

## Key Questions
- Which stations see the highest ridership, and has that changed since 2019?
- What are the peak travel times, and do they vary by borough?
- How has ridership recovered post-pandemic across different parts of the system?
- Are there underserved stations where ridership growth outpaces service frequency?

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
from datetime import datetime, timedelta

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print("Setup complete.")

## 2. Data Acquisition

The MTA publishes turnstile data weekly at [http://web.mta.info/developers/turnstile.html](http://web.mta.info/developers/turnstile.html). Each file contains one week of audit records.

Fields:
- **C/A, UNIT, SCP**: Identifiers for the control area, remote unit, and specific turnstile
- **STATION, LINENAME**: Station name and subway lines served
- **DATE, TIME**: Audit timestamp
- **ENTRIES, EXITS**: Cumulative counter readings

In [ ]:
# Download recent weeks of turnstile data
# The MTA URL pattern: http://web.mta.info/developers/data/nyct/turnstile/turnstile_YYMMDD.txt

def get_turnstile_data(weeks_back=4):
    """Download the most recent N weeks of MTA turnstile data."""
    
    base_url = "http://web.mta.info/developers/data/nyct/turnstile/turnstile_{}.txt"
    frames = []
    
    # MTA files are published on Saturdays
    today = datetime.now()
    # Find the most recent Saturday
    days_since_saturday = (today.weekday() + 2) % 7
    last_saturday = today - timedelta(days=days_since_saturday)
    
    for i in range(weeks_back):
        date = last_saturday - timedelta(weeks=i)
        date_str = date.strftime('%y%m%d')
        url = base_url.format(date_str)
        
        try:
            print(f"Downloading {url}...")
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            df = pd.read_csv(io.StringIO(resp.text))
            df.columns = df.columns.str.strip()
            frames.append(df)
            print(f"  -> {len(df):,} records")
        except Exception as e:
            print(f"  -> Skipped: {e}")
    
    if frames:
        data = pd.concat(frames, ignore_index=True)
        print(f"\nTotal records: {len(data):,}")
        return data
    else:
        raise ValueError("No data downloaded")

raw = get_turnstile_data(weeks_back=4)
raw.head()

## 3. Data Cleaning

The raw data requires careful cleaning:
- Counter values are **cumulative**, so we need to compute deltas between consecutive readings
- Counters occasionally **reset**, producing enormous negative or positive deltas that need to be capped
- Some turnstiles have **duplicate or missing** audit records

In [ ]:
def clean_turnstile_data(df):
    """Clean MTA turnstile data and compute per-period entry/exit counts."""
    
    df = df.copy()
    
    # Parse datetime
    df['DATETIME'] = pd.to_datetime(df['DATE'] + ' ' + df['TIME'], format='%m/%d/%Y %H:%M:%S')
    
    # Create a unique turnstile identifier
    df['TURNSTILE_ID'] = df['C/A'] + '_' + df['UNIT'] + '_' + df['SCP'] + '_' + df['STATION']
    
    # Sort by turnstile and time
    df = df.sort_values(['TURNSTILE_ID', 'DATETIME']).reset_index(drop=True)
    
    # Compute entry/exit deltas within each turnstile
    df['PREV_ENTRIES'] = df.groupby('TURNSTILE_ID')['ENTRIES'].shift(1)
    df['PREV_EXITS'] = df.groupby('TURNSTILE_ID')['EXITS'].shift(1)
    
    df['ENTRY_DELTA'] = df['ENTRIES'] - df['PREV_ENTRIES']
    df['EXIT_DELTA'] = df['EXITS'] - df['PREV_EXITS']
    
    # Drop first reading per turnstile (no delta possible)
    df = df.dropna(subset=['ENTRY_DELTA'])
    
    # Cap unreasonable values (counter resets, errors)
    # A single turnstile shouldn't see more than ~10,000 entries in a 4-hour window
    MAX_COUNT = 10_000
    df = df[(df['ENTRY_DELTA'] >= 0) & (df['ENTRY_DELTA'] <= MAX_COUNT)]
    df = df[(df['EXIT_DELTA'] >= 0) & (df['EXIT_DELTA'] <= MAX_COUNT)]
    
    # Extract time features
    df['HOUR'] = df['DATETIME'].dt.hour
    df['DAY_OF_WEEK'] = df['DATETIME'].dt.day_name()
    df['DATE_ONLY'] = df['DATETIME'].dt.date
    df['IS_WEEKEND'] = df['DATETIME'].dt.dayofweek >= 5
    
    print(f"Cleaned data: {len(df):,} records")
    print(f"Stations: {df['STATION'].nunique()}")
    print(f"Date range: {df['DATETIME'].min().date()} to {df['DATETIME'].max().date()}")
    
    return df

df = clean_turnstile_data(raw)
df.head()

## 4. Ridership Patterns

### 4.1 Daily Ridership Trends

In [ ]:
# Daily total entries across the system
daily = df.groupby('DATE_ONLY')['ENTRY_DELTA'].sum().reset_index()
daily.columns = ['date', 'total_entries']
daily['date'] = pd.to_datetime(daily['date'])

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(daily['date'], daily['total_entries'] / 1e6, color='#2E86AB', alpha=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Total Entries (millions)')
ax.set_title('NYC Subway Daily Ridership', fontsize=16, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../assets/daily_ridership.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Average daily entries: {daily['total_entries'].mean():,.0f}")
print(f"Peak day: {daily.loc[daily['total_entries'].idxmax(), 'date'].strftime('%A, %B %d')} "
      f"({daily['total_entries'].max():,.0f} entries)")

### 4.2 Ridership by Hour of Day

In [ ]:
# Entries by hour, split by weekday vs weekend
hourly = df.groupby(['HOUR', 'IS_WEEKEND'])['ENTRY_DELTA'].mean().reset_index()
hourly['day_type'] = hourly['IS_WEEKEND'].map({False: 'Weekday', True: 'Weekend'})

fig, ax = plt.subplots(figsize=(12, 5))
for day_type, group in hourly.groupby('day_type'):
    ax.plot(group['HOUR'], group['ENTRY_DELTA'], marker='o', linewidth=2.5, 
            label=day_type, markersize=6)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Entries per Turnstile per Period')
ax.set_title('Ridership by Hour: Weekday vs Weekend', fontsize=16, fontweight='bold')
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=45)
ax.legend(fontsize=12)
ax.axvspan(7, 10, alpha=0.1, color='red', label='AM Peak')
ax.axvspan(16, 19, alpha=0.1, color='orange', label='PM Peak')
plt.tight_layout()
plt.savefig('../assets/hourly_ridership.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Busiest Stations

In [ ]:
# Top 20 stations by total entries
station_totals = df.groupby('STATION')['ENTRY_DELTA'].sum().sort_values(ascending=False)
top_20 = station_totals.head(20).reset_index()
top_20.columns = ['station', 'total_entries']

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(top_20)), top_20['total_entries'] / 1e6, color='#2E86AB', alpha=0.85)
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['station'], fontsize=11)
ax.set_xlabel('Total Entries (millions)', fontsize=12)
ax.set_title('Top 20 Busiest NYC Subway Stations', fontsize=16, fontweight='bold')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, top_20['total_entries']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.2f}M', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../assets/top_stations.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Day-of-Week Patterns

In [ ]:
# Ridership by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_dow = df.groupby('DAY_OF_WEEK')['ENTRY_DELTA'].sum().reindex(day_order).reset_index()
daily_dow.columns = ['day', 'total_entries']

colors = ['#2E86AB'] * 5 + ['#A23B72'] * 2  # Blue for weekdays, purple for weekends

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(daily_dow['day'], daily_dow['total_entries'] / 1e6, color=colors, alpha=0.85)
ax.set_ylabel('Total Entries (millions)')
ax.set_title('Ridership by Day of Week', fontsize=16, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../assets/day_of_week.png', dpi=150, bbox_inches='tight')
plt.show()

# Weekend vs weekday ratio
weekday_avg = daily_dow[daily_dow['day'].isin(day_order[:5])]['total_entries'].mean()
weekend_avg = daily_dow[daily_dow['day'].isin(day_order[5:])]['total_entries'].mean()
print(f"Weekend ridership is {weekend_avg/weekday_avg:.0%} of weekday ridership")

## 5. Interactive Station Map

Using Folium to create an interactive map showing ridership volume at each station. Larger circles = more riders.

*Note: Station coordinates are approximated from the MTA's station list. For production analysis, use the official GTFS stops.txt file.*

In [ ]:
import folium

# We'll need station coordinates — download from MTA GTFS or use a lookup
# For now, create the map framework. In practice, merge with GTFS stops.txt

# Station coordinates can be obtained from:
# http://web.mta.info/developers/data/nyct/subway/Stations.csv

try:
    stations_url = "http://web.mta.info/developers/data/nyct/subway/Stations.csv"
    stations_geo = pd.read_csv(stations_url)
    stations_geo.columns = stations_geo.columns.str.strip()
    
    # Get the top stations with coordinates
    station_rides = df.groupby('STATION')['ENTRY_DELTA'].sum().reset_index()
    station_rides.columns = ['station', 'total_entries']
    
    print(f"Station geo columns: {stations_geo.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(stations_geo.head())
    
except Exception as e:
    print(f"Could not load station coordinates: {e}")
    print("You can download them manually from the MTA developer portal.")
    print("Skipping map for now — will revisit with GTFS data.")

In [ ]:
# Create interactive map (will work once station coordinates are merged)
# This is the framework — we'll refine once we confirm the coordinate data format

try:
    # Create base map centered on NYC
    nyc_map = folium.Map(location=[40.7128, -74.0060], zoom_start=11,
                         tiles='CartoDB positron')
    
    # If we have coordinates, add circle markers
    # (This section will be updated based on the actual column names)
    
    print("Map framework created. Will add station markers once coordinates are confirmed.")
    print("Save this as an HTML file to view interactively:")
    print("  nyc_map.save('../maps/nyc_ridership_map.html')")
    
except Exception as e:
    print(f"Map creation: {e}")

## 6. Key Findings & Next Steps

### Findings
*(To be filled in after running the analysis with real data)*

- **Peak hours**: ...
- **Busiest stations**: ...
- **Weekend vs. weekday**: ...
- **Recovery trends**: ...

### Next Steps
- Merge with GTFS data for accurate station coordinates and service frequency
- Compare current ridership to pre-pandemic (2019) levels by station
- Analyze ridership density per route mile to identify capacity constraints
- Build an interactive dashboard for exploring patterns by borough and line
- Cross-reference with Census data to examine equity dimensions of service coverage

### Data Notes
- Turnstile counters are cumulative and can reset, requiring careful delta computation
- Some stations have multiple complexes that may appear as separate entries
- The 4-hour audit interval means exact rush-hour peaks may be smoothed out

---
*This analysis is part of [Aidan Moran's Transit Analytics Portfolio](https://github.com/aidanmoran). View the full project on GitHub.*